In [25]:
import pandas as pd
import numpy as np

# Load cleaned data
cleaned_df = pd.read_csv('../datasets/interim/cleaned_train.csv')
print(f"Train set shape: {cleaned_df.shape}")

Train set shape: (20804, 58)


In [26]:
# -- FILLING MISSING DATA --
# We can't fill every missing data with imputation. For example "listing_url" column can not be filled.
# After analyzing the attributes, we determined the ones that can be filled with numerical values
numeric_fill_cols = [
    'accommodates', 'bedrooms', 'beds', 'minimum_nights', 'maximum_nights',
    'host_listings_count', 'host_total_listings_count',
    'calculated_host_listings_count',
    'review_scores_rating', 'review_scores_accuracy', 'review_scores_cleanliness',
    'review_scores_checkin', 'review_scores_communication',
    'review_scores_location', 'review_scores_value'
]

# Filling these columns with median values (to not be sensitive to outliers)
for col in numeric_fill_cols:
    if col in cleaned_df.columns:
        cleaned_df[col] = cleaned_df[col].fillna(cleaned_df[col].median())

In [27]:
# Filling categorical attributes with mode if missing rows are less than %10 (otherwise filling with mode wouldnt be healthy)
categorical_cols = [
    'host_is_superhost', 'host_has_profile_pic', 'host_identity_verified',
    'room_type', 'property_type', 'instant_bookable', 'host_response_time'
]
threshold = 0.10

for col in categorical_cols:
    if col in cleaned_df.columns:
        missing_ratio = cleaned_df[col].isnull().mean()

        if missing_ratio > threshold:
            # Case 1: High missing ratio -> Fill with 'Unknown'
            # Convert to string first to handle mixed types
            cleaned_df[col] = cleaned_df[col].astype(str)
            cleaned_df[col] = cleaned_df[col].replace('nan', 'Unknown').fillna('Unknown')
            print(f"[{col}] Missing: {missing_ratio*100:.1f}% (>10%) -> Filled with 'Unknown'")

        elif missing_ratio > 0:
            # Case 2: Low missing ratio -> Fill with Mode
            mode_val = cleaned_df[col].mode()[0]
            cleaned_df[col] = cleaned_df[col].fillna(mode_val)
            print(f"[{col}] Missing: {missing_ratio*100:.1f}% (<10%) -> Filled with Mode '{mode_val}'")

[host_is_superhost] Missing: 2.2% (<10%) -> Filled with Mode 'f'
[host_has_profile_pic] Missing: 2.5% (<10%) -> Filled with Mode 't'
[host_identity_verified] Missing: 2.5% (<10%) -> Filled with Mode 't'
[host_response_time] Missing: 30.5% (>10%) -> Filled with 'Unknown'


In [28]:
# --- SAFETY NET ---
# Fill ANY remaining object columns with 'Unknown' to prevent model crash
remaining_cat_cols = cleaned_df.select_dtypes(include=['object']).columns
cleaned_df[remaining_cat_cols] = cleaned_df[remaining_cat_cols].fillna('Unknown')

# Fill ANY remaining numerical columns with 0
remaining_num_cols = cleaned_df.select_dtypes(include=[np.number]).columns
cleaned_df[remaining_num_cols] = cleaned_df[remaining_num_cols].fillna(0)

# FINAL CHECK
print("-" * 30)
print(f"Imputation Complete. Total Remaining NaNs: {cleaned_df.isnull().sum().sum()}")

------------------------------
Imputation Complete. Total Remaining NaNs: 0


In [29]:
# Save the filled data to a new CSV file
output_path = '../datasets/interim/filled_cleaned_train.csv'

# index=False is important to avoid creating an extra "Unnamed: 0" column
cleaned_df.to_csv(output_path, index=False)